In [1]:
import io
import os
import re
import fitz
import uuid
import litellm
import chromadb
from dotenv import load_dotenv
from chromadb.config import Settings
from sentence_transformers import SentenceTransformer

load_dotenv()

LLM_API_KEY = os.getenv("GROQ_API_KEY")

c:\Users\HP\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
class Config:
    def __init__(self, **kwargs):
        self.embedding_model = kwargs.get("embedding_model")
        self.llm_model = kwargs.get("llm_model")

        self.chunk_size = kwargs.get("chunk_size")
        self.chunk_overlap = kwargs.get("chunk_overlap")

        self.top_k = kwargs.get("top_k")

        self.vector_db_path = kwargs.get("vector_db_path")
        self.collection_name = kwargs.get("collection_name")

        self.temperature = kwargs.get("temperature")
        self.max_tokens = kwargs.get("max_tokens")

        self.llm_api_key = os.getenv(kwargs.get("llm_api_key", "GROQ_API_KEY")) 
    
    def validate(self):
        if not self.llm_api_key:
            raise ValueError("LLM_API_KEY not found in environment variables")
        
        if not self.embedding_model:
            raise ValueError("Embedding model must be provided")

        if not self.llm_model:
            raise ValueError("LLM model must be provided")

        if not self.chunk_size:
            raise ValueError("Chunk size must be provided")

        if not self.top_k:
            raise ValueError("top_k must be provided")
    
    def to_dict(self):
        return {
            "embedding_model": self.embedding_model,
            "llm_model": self.llm_model,
            "chunk_size": self.chunk_size,
            "chunk_overlap": self.chunk_overlap,
            "top_k": self.top_k,
            "vector_db_path": self.vector_db_path,
            "collection_name": self.collection_name,
            "temperature": self.temperature,
            "max_tokens": self.max_tokens,
        }

In [3]:
class DocumentLoader:
    def __init__(self, pdf_bytes: bytes):
        self.pdf_bytes = pdf_bytes

    def load_pdf(self):
        pdf_stream = io.BytesIO(self.pdf_bytes)
        doc = fitz.open(stream=pdf_stream, filetype="pdf")
        pages_text = []
        for page_number in range(len(doc)):
            page = doc.load_page(page_number)
            text = page.get_text() 
            pages_text.append({
                "page": page_number + 1,
                "text": text
            })
        doc.close()
        return pages_text

In [4]:
class TextSplitter:
    def __init__(self, chunk_size: int, chunk_overlap: int):
        self.chunk_size = chunk_size
        self.chunk_overlap = chunk_overlap
        self.separators = [
            "\n\n",
            "\n",
            ". ",
            " ",
            ""
        ]

    def split_documents(self, documents):
        all_chunks = []
        for doc in documents:
            page = doc.get("page")
            text = doc.get("text", "")
            chunks = self.split_text(text)
            for chunk in chunks:
                all_chunks.append({
                    "page": page,
                    "text": chunk
                })
        return all_chunks

    def split_text(self, text: str):
        splits = self._recursive_split(text, self.separators)
        return self._apply_overlap(splits)

    def _recursive_split(self, text, separators):
        if len(text) <= self.chunk_size:
            return [text]
        if not separators:
            return self._force_split(text)
        separator = separators[0]
        if separator == "":
            pieces = list(text)
        else:
            pieces = text.split(separator)
        chunks = []
        current_chunk = ""
        for piece in pieces:
            if separator != "":
                piece = piece + separator
            if len(current_chunk) + len(piece) <= self.chunk_size:
                current_chunk += piece
            else:
                if current_chunk:
                    chunks.append(current_chunk.strip())
                if len(piece) > self.chunk_size:
                    sub_chunks = self._recursive_split(piece, separators[1:])
                    chunks.extend(sub_chunks)
                    current_chunk = ""
                else:
                    current_chunk = piece
        if current_chunk:
            chunks.append(current_chunk.strip())
        return chunks

    def _force_split(self, text):
        chunks = []
        for i in range(0, len(text), self.chunk_size):
            chunks.append(text[i:i + self.chunk_size])
        return chunks

    def _apply_overlap(self, chunks):
        if self.chunk_overlap == 0:
            return chunks
        overlapped_chunks = []
        for i, chunk in enumerate(chunks):
            if i == 0:
                overlapped_chunks.append(chunk)
                continue
            overlap_text = chunks[i - 1][-self.chunk_overlap:]
            new_chunk = overlap_text + chunk
            overlapped_chunks.append(new_chunk)
        return overlapped_chunks

In [5]:
class Embed:
    def __init__(self, model_name: str):
        self.model_name = model_name
        self.model = SentenceTransformer(model_name)

    def embed_documents(self, chunks):
        texts = [chunk["text"] for chunk in chunks]
        embeddings = self.model.encode(
            texts,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        results=[]
        for chunk, emb in zip(chunks, embeddings):
            results.append({
                "page": chunk["page"],
                "text": chunk["text"],
                "embedding": emb
            })
        return results

    def embed_query(self, query: str):
        embedding = self.model.encode(
            query,
            convert_to_numpy=True,
            normalize_embeddings=True
        )
        return embedding

In [6]:
class VectorStore:
    def __init__(self, persist_directory: str, collection_name: str):
        self.persist_directory = persist_directory
        self.collection_name = collection_name

        self.client = chromadb.Client(
            Settings(persist_directory=persist_directory)
        )

        self.collection = self.client.get_or_create_collection(
            name=collection_name
        )

    def add_embeddings(self, embeddings, metadata):
        ids = [str(uuid.uuid4()) for _ in metadata]
        documents = [m["text"] for m in metadata]

        self.collection.add(
            ids=ids,
            embeddings=embeddings,
            documents=documents,
            metadatas=metadata
        )

    def save(self):
        self.client.persist()

    def load(self):
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name
        )

    def similarity_search(self, query_embedding, top_k: int):
        results = self.collection.query(
            query_embeddings=[query_embedding],
            n_results=top_k
        )

        documents = results["documents"][0]
        metadatas = results["metadatas"][0]
        distances = results["distances"][0]

        retrieved = []

        for doc, meta, dist in zip(documents, metadatas, distances):
            retrieved.append({
                "text": doc,
                "metadata": meta,
                "distance": dist
            })
        return retrieved

In [7]:
class Retriever:
    def __init__(self, vector_store, embed):
        self.vector_store = vector_store
        self.embed = embed
    
    def retrieve(self, query: str, top_k: int):
        query_embedding = self.embed.embed_query(query)

        results = self.vector_store.similarity_search(
            query_embedding=query_embedding,
            top_k=top_k,
        )
        return results

In [8]:
class PromptBuilder:
    def __init__(self):
        self.template = """
You are a helpful AI assistant.

Use ONLY the provided context to answer the question.
If the answer is not present in the context, say:
"I could not find the answer in the provided documents."

Context:
{context}

Question:
{question}

Answer:
"""

    def build_prompt(self, query: str, context_chunks):
        context = "\n\n".join(
            [chunk["text"] for chunk in context_chunks]
        )
        prompt = self.template.format(
            context=context,
            question=query
        )
        return prompt

In [9]:
class LLMGenerator:
    def __init__(self, model_name: str, temperature: float = 0.0, max_tokens: int = 500):
        self.model_name = model_name
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate(self, prompt: str):
        response = litellm.completion(
            model=self.model_name,
            messages=[{
                "role": "user",
                "content": prompt
            }],
            temperature=self.temperature,
            max_tokens=self.max_tokens,
        )
        #return response
        return response["choices"][0]["message"]["content"]

In [10]:
class QueryExpander:
    def __init__(self, generator, num_queries=4):
        self.generator = generator
        self.num_queries = num_queries

    def expand(self, query: str):
        prompt = f"""
Generate {self.num_queries} different search queries that capture different perspectives of the question.

Original Question:
{query}

Queries:
"""
        response = self.generator.generate(prompt)

        queries = [q.strip("- ").strip() for q in response.split("\n") if q.strip()]
        return queries[:self.num_queries]

In [ ]:
class RAGFusionPipeline:
    def __init__(self, retriever, prompt_builder, generator):
        self.retriever = retriever
        self.prompt_builder = prompt_builder
        self.generator = generator
        self.query_expander = QueryExpander(generator)

    def reciprocal_rank_fusion(self, results_list, k=60):
        scores = {}
        for results in results_list:
            for rank, doc in enumerate(results):
                doc_id = doc["text"]
                if doc_id not in scores:
                    scores[doc_id] = 0
                scores[doc_id] += 1 / (k + rank + 1)

        reranked = sorted(scores.items(), key=lambda x: x[1], reverse=True)
        fused_results = []
        seen = set()
        for doc_text, score in reranked:
            if doc_text not in seen:
                fused_results.append({
                    "text": doc_text,
                    "score": score
                })
                seen.add(doc_text)
        return fused_results

    def run(self, query: str, top_k: int = 5):
        queries = self.query_expander.expand(query)
        queries.append(query)
        all_results = []

        for q in queries:
            results = self.retriever.retrieve(q, top_k)
            all_results.append(results)

        fused_results = self.reciprocal_rank_fusion(all_results)

        final_chunks = fused_results[:top_k]
        final_chunks_formatted = [
            {"text": doc["text"]}
            for doc in final_chunks
        ]

        prompt = self.prompt_builder.build_prompt(
            query=query,
            context_chunks=final_chunks_formatted
        )
        answer = self.generator.generate(prompt)
        return {
            "query": query,
            "answer": answer,
            "expanded_queries": queries,
            "sources": final_chunks
        }

In [12]:
pdf_path = "rag_sample.pdf"
with open(pdf_path, "rb") as f:
    pdf_bytes = f.read()

loader = DocumentLoader(pdf_bytes=pdf_bytes)
docs = loader.load_pdf()
docs

[{'page': 1,
  'text': 'THE HISTORY OF SPACE EXPLORATION: 1945 TO PRESENT AND BEYOND \nSECTION 1: POST-WORLD WAR II ROCKETRY FOUNDATIONS \n1.1 THE CAPTURE OF AEROSPACE TECHNOLOGY\u200b\nThe cessation of global hostilities in 1945 marked the immediate commencement of a \nclandestine and highly competitive effort by the Allied powers to secure the remnants of the \nGerman V-2 ballistic missile program. This era established the foundational aerospace \nengineering principles that would dictate the trajectory of space exploration for the next eight \ndecades. The V-2, originally designated the A4, was the first artificial object to cross the \nKarman line, the internationally recognized boundary of space situated at an altitude of 100 \nkilometers. The propulsion systems utilized a combination of liquid oxygen, hereafter \nreferred to as LOX, and an ethanol-water mixture, establishing the baseline for cryogenic \nliquid rocket engines. \n1.1.1 OPERATION PAPERCLIP AND SOVIET ASSIMILATION\u2

In [13]:
splitter = TextSplitter(chunk_size=500, chunk_overlap=50)
chunks = splitter.split_documents(documents=docs)
len(chunks)

86

In [14]:
embed = Embed(model_name="all-MiniLM-L6-v2")
embeddings = embed.embed_documents(chunks=chunks)
len(embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 236.96it/s, Materializing param=pooler.dense.weight]                             
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


86

In [15]:
vector_store = VectorStore(persist_directory="", collection_name="p4_rag_from_scratch")
vector_store.add_embeddings(
    [e["embedding"] for e in embeddings],
    [{"page": e["page"], "text": e["text"]} for e in embeddings]
)

In [16]:
retriever = Retriever(vector_store=vector_store, embed=embed)

prompt_builder = PromptBuilder()
generator = LLMGenerator("groq/llama-3.3-70b-versatile")

rag = RAGFusionPipeline(
    retriever=retriever,
    prompt_builder=prompt_builder,
    generator=generator
)

response = rag.run("When was voyager 1 launched?")
response

{'query': 'When was voyager 1 launched?',
 'answer': 'Voyager 1 was launched on September 5, 1977.',
 'expanded_queries': ['Here are 4 different search queries that capture different perspectives of the question:',
  '1. **Simple factual query**: "Voyager 1 launch date"',
  '2. **Historical context query**: "NASA Voyager 1 mission timeline"',
  '3. **Technical specification query**: "Voyager 1 spacecraft launch details"',
  'When was voyager 1 launched?'],
 'sources': [{'text': "Voyager 1 provided unprecedented data on the Jovian and Saturnian systems, discovering \nactive volcanism on Jupiter's moon Io and the complex ring dynamics of Saturn. In 2012, \nVoyager 1 crossed the heliopause, becoming the first human-made object to enter \ninterstellar space. The communication delay with Voyager 1 is immense; a signal traveling at \nthe speed of light takes over 22 hours to reach the spacecraft. \n6.3 CASE STUDY 3: CASSINI-HUYGENS\u200b",
   'score': 0.06478893337698204},
  {'text': 'ge 21\